# Haiku Linear Probing Accuracy

**Project Name:** Haiku

## Purpose
- Publication-ready notebook for reproducible evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR   = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
METADATA_DIR     = Path('<PATH_TO_REGION_METADATA>')
SAMPLES_JSON     = HAIKU_ROOT / 'overlap_samples_final.json'
TEST_REGIONS_TXT = Path('<PATH_TO_test_regions.txt>')
OUTPUT_DIR       = HAIKU_ROOT / 'downstream' / 'figs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)


In [ ]:
%load_ext cuml.accel

In [ ]:
import json
import pandas as pd


sample_dict = json.load(open(SAMPLES_JSON))
sample_ids = list(sample_dict.keys())


with open(TEST_REGIONS_TXT, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

sample_ids = list(set(test_ids) & set(sample_ids))

ref_ids = sorted(sample_ids)


In [ ]:

import os
from tqdm import tqdm

region_metadata_dir = str(METADATA_DIR)

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
import torch

he_embedding = torch.load(EMBEDDINGS_DIR / 'he_embedding.pt')
codex_embedding = torch.load(EMBEDDINGS_DIR / 'codex_embedding.pt')
region_label = torch.load(EMBEDDINGS_DIR / 'region_label.pt')
virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'virtual_codex_embedding.pt')
text_embedding = torch.load(EMBEDDINGS_DIR / 'text_embedding.pt')
musk_he_embedding = torch.load(EMBEDDINGS_DIR / 'musk_he_embedding.pt')


In [ ]:
from tqdm import tqdm
import numpy as np

ref_ids = sorted(sample_ids)

metadata_dict_values = {}
metadata_dict_ids = {}

keys = ['tissue_type', 'grade', 'tnm', 'type']

for key in keys:
    metadata_dict_values[key] = []
    metadata_dict_ids[key] = []

print(f"Processing {len(region_label)} samples for metadata lookup...")
for i, sample in tqdm(enumerate(region_label), total=len(region_label), desc="Processing metadata"):
    #patch_id = sample['patch_id']
    #print(sample)
    region_id = ref_ids[sample].split('_')[0]
    df = region_metadata.get(region_id, None)
    if df is not None:
        for key in keys:
            if key in df['FEATURE_NAME'].values:
                if (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'nan') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'Unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == np.nan):
                    continue
                else:
                    metadata_dict_values[key].append(df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0])
                    metadata_dict_ids[key].append(i)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

def linear_probe_kfold(
    embeddings,
    labels,
    n_splits=5,
    s=5,
    random_state=42,
    Cs=None,
    select_by="f1_macro"  # one of {"f1_macro", "f1_micro", "accuracy"}
):
    """
    Perform linear probing with logistic regression using K-fold cross-validation,
    grid-searching over C (default 5 values) and picking the best C by mean metric.

    Returns the SAME flat dict as the previous version:
        {
            "f1_macro_mean", "f1_macro_std",
            "f1_micro_mean", "f1_micro_std",
            "accuracy_mean", "accuracy_std",
            "f1_macros", "f1_micros", "accuracies"
        }
    """
    # Convert tensors → numpy
    if hasattr(embeddings, "cpu"):
        embeddings = embeddings.cpu().numpy()
    if hasattr(labels, "cpu"):
        labels = labels.cpu().numpy()
    embeddings = np.asarray(embeddings)
    labels = np.asarray(labels)

    # Default 5-point C grid if not given
    if Cs is None:
        Cs = np.logspace(-2, 2, 5)  # [0.01, 0.1, 1, 10, 100]
    Cs = np.asarray(Cs, dtype=float)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Evaluate each C with the same outer CV
    summaries = {}
    for C in Cs:
        f1_macros, f1_micros, accuracies = [], [], []
        for train_idx, test_idx in skf.split(embeddings, labels):
            X_train, X_test = embeddings[train_idx], embeddings[test_idx]
            y_train, y_test = labels[train_idx], labels[test_idx]

            clf = LogisticRegression(
                C=C,
                random_state=42,
                max_iter=50000,
                solver="lbfgs",
                multi_class="auto",
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            f1_macros.append(f1_score(y_test, y_pred, average='macro', zero_division=0))
            f1_micros.append(f1_score(y_test, y_pred, average='micro', zero_division=0))
            accuracies.append(accuracy_score(y_test, y_pred))

        summaries[float(C)] = {
            "f1_macro_mean": np.mean(f1_macros),
            "f1_macro_std":  np.std(f1_macros),
            "f1_micro_mean": np.mean(f1_micros),
            "f1_micro_std":  np.std(f1_micros),
            "accuracy_mean": np.mean(accuracies),
            "accuracy_std":  np.std(accuracies),
            "f1_macros": f1_macros,
            "f1_micros": f1_micros,
            "accuracies": accuracies,
        }

    # Pick best C by requested metric
    key_mean = f"{select_by}_mean"
    best_C = max(summaries.keys(), key=lambda c: summaries[c][key_mean])
    best_res = summaries[best_C]

    # Pretty print
    print("=== Linear Probe K-Fold with C Grid Search ===")
    print(f"C candidates: {list(map(float, Cs))}")
    print(f"Selection metric: {select_by} (mean across folds)\n")
    for C in Cs:
        r = summaries[float(C)]
        print(f"C={float(C):g}  |  "
              f"F1 Macro: {r['f1_macro_mean']:.4f} ± {r['f1_macro_std']:.4f}  |  "
              f"F1 Micro: {r['f1_micro_mean']:.4f} ± {r['f1_micro_std']:.4f}  |  "
              f"Acc: {r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f}")
    print("\n--- Best ---")
    print(f"Best C: {best_C:g}")
    print(f"F1 Macro: {best_res['f1_macro_mean']:.4f} ± {best_res['f1_macro_std']:.4f}")
    print(f"F1 Micro: {best_res['f1_micro_mean']:.4f} ± {best_res['f1_micro_std']:.4f}")
    print(f"Accuracy: {best_res['accuracy_mean']:.4f} ± {best_res['accuracy_std']:.4f}")

    # Return EXACTLY like previous
    results = {
        "f1_macro_mean": best_res["f1_macro_mean"],
        "f1_macro_std":  best_res["f1_macro_std"],
        "f1_micro_mean": best_res["f1_micro_mean"],
        "f1_micro_std":  best_res["f1_micro_std"],
        "accuracy_mean": best_res["accuracy_mean"],
        "accuracy_std":  best_res["accuracy_std"],
        "f1_macros":     best_res["f1_macros"],
        "f1_micros":     best_res["f1_micros"],
        "accuracies":    best_res["accuracies"],
    }

    print("=== Linear Probe Results (selected C) ===")
    print(f"F1 Macro: {results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}")
    print(f"F1 Micro: {results['f1_micro_mean']:.4f} ± {results['f1_micro_std']:.4f}")
    print(f"Accuracy: {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}")
    return results

def majority_vote_baseline(labels, n_splits=5, random_state=42):
    """
    Majority Voting Baseline: for each fold, predict the most frequent class
    in the training split for all test examples.

    Returns:
        {
            "f1_macro_mean", "f1_macro_std",
            "f1_micro_mean", "f1_micro_std",
            "accuracy_mean", "accuracy_std",
            "f1_macros", "f1_micros", "accuracies"
        }
    """
    if hasattr(labels, "cpu"):
        labels = labels.cpu().numpy()
    labels = np.asarray(labels)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    f1_macros, f1_micros, accuracies = [], [], []
    for train_idx, test_idx in skf.split(np.zeros_like(labels), labels):
        y_train, y_test = labels[train_idx], labels[test_idx]
        vals, counts = np.unique(y_train, return_counts=True)
        majority_class = vals[np.argmax(counts)]
        majority_pred = np.full_like(y_test, majority_class)
        f1_macros.append(f1_score(y_test, majority_pred, average='macro', zero_division=0))
        f1_micros.append(f1_score(y_test, majority_pred, average='micro', zero_division=0))
        accuracies.append(accuracy_score(y_test, majority_pred))
    results = {
        "f1_macro_mean": np.mean(f1_macros),
        "f1_macro_std":  np.std(f1_macros),
        "f1_micro_mean": np.mean(f1_micros),
        "f1_micro_std":  np.std(f1_micros),
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std":  np.std(accuracies),
        "f1_macros": f1_macros,
        "f1_micros": f1_micros,
        "accuracies": accuracies,
    }
    print("=== Majority Voting Baseline ===")
    print(f"F1 Macro: {results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}")
    print(f"F1 Micro: {results['f1_micro_mean']:.4f} ± {results['f1_micro_std']:.4f}")
    print(f"Accuracy: {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}")
    return results


In [ ]:
np.unique(metadata_dict_values['grade'])

In [ ]:
np.unique(metadata_dict_values['type'])

In [ ]:
np.unique(metadata_dict_values['tnm'])

In [ ]:
category_map = {}

category_map['type'] = {
    'normal': 'Normal',
    'nat': 'Normal',
    'at': 'Normal',
    'AT': 'Normal',
    "NAT": 'Normal',
    'hyperplasia': 'Benign/Precancerous',
    'malignant': 'Primary Tumor',
    'tumor primary': 'Primary Tumor',
    'Tumor Primary': 'Primary Tumor',
    'Maglignant': 'Primary Tumor',
    'metastasis': 'Metastatic Tumor',
    'nan': None,
    '-': None,
    '*': None
}

category_map['grade'] = {
    '1': 'G1',
    '1--2': 'G1',
    'g1': 'G1',
    '2': 'G2',
    '2--3': 'G2',
    'g2': 'G2',
    '3': 'G3',
    'g3': 'G3',
    'nan': None,
    '-': None,
    '*': None
}


In [ ]:
keys = ['tissue_type', 'grade', 'tnm', 'type']


def clean_labels(labels, map):
    """
    Map various survival status labels to 'alive' or 'death'.
    """
    mapped = []
    for x in labels:
        if x in map:
            mapped.append(map[x])
        else:
            mapped.append(x)

    return np.array(mapped)


for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and str(val).lower() != '-'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


for key in metadata_dict_values:
    if key in category_map:
        filtered_values = clean_labels(metadata_dict_values[key], category_map[key])
        metadata_dict_ids[key] = np.array(metadata_dict_ids[key])[filtered_values != None]
        metadata_dict_values[key] = np.array(filtered_values)[filtered_values != None]


In [ ]:
import re

def split_tnm(values):
    """
    Split TNM strings like 'T3N1bM0' into separate labels.
    Filters out '-', 'nan', 'unknown', empty, None.

    Parameters
    ----------
    values : Sequence[str]
        Array/list of TNM strings.

    Returns
    -------
    result : dict
        {
          'indices': [idx_of_each_valid_entry],
          'T': ['T1', 'T1a', 'T4B', ...],
          'N': ['N0', 'N1b', ...],
          'M': ['M0', 'M1', ...]
        }
    """
    clean_exclude = {'-', '', 'nan', 'none', 'unknown'}
    # Capture contiguous T..., N..., M... chunks (letters+digits), in order.
    pat = re.compile(r'^(T[0-9A-Za-z]+)(N[0-9A-Za-z]+)(M[0-9A-Za-z]+)$')

    idxs, Ts, Ns, Ms = [], [], [], []
    for i, v in enumerate(values):
        if v is None:
            continue
        s = str(v).strip()
        if s.lower() in clean_exclude:
            continue
        s = s.replace(' ', '')  # ensure no spaces like 'T3 N1 M0'
        m = pat.match(s)
        if not m:
            # Skip anything that doesn't look like proper TNM
            continue
        t, n, mstage = m.group(1), m.group(2), m.group(3)
        idxs.append(i)
        Ts.append(t)
        Ns.append(n)
        Ms.append(mstage)

    return {'indices': idxs, 'T': Ts, 'N': Ns, 'M': Ms}


res = split_tnm(metadata_dict_values['tnm'])

# assuming you want new keys 'T', 'N', 'M' parallel to 'tnm'
metadata_dict_values['T'] = np.array(res['T'])
metadata_dict_values['N'] = np.array(res['N'])
metadata_dict_values['M'] = np.array(res['M'])

# Map split_tnm positions back to original embedding indices via metadata_dict_ids['tnm']
tnm_ids = np.array(metadata_dict_ids['tnm'])
metadata_dict_ids['T'] = tnm_ids[res['indices']]
metadata_dict_ids['N'] = tnm_ids[res['indices']]
metadata_dict_ids['M'] = tnm_ids[res['indices']]

In [ ]:
cancer_indices = metadata_dict_ids['type'][(metadata_dict_values['type'] != 'Normal') & (metadata_dict_values['type'] != 'Benign/Precancerous')]

In [ ]:
np.unique(metadata_dict_values['type'][np.isin(metadata_dict_ids['type'], cancer_indices)])

In [ ]:
np.unique(metadata_dict_values['T'][np.isin(metadata_dict_ids['T'], cancer_indices)])

In [ ]:
print(np.unique(metadata_dict_values['grade'][np.isin(metadata_dict_ids['grade'], cancer_indices)]))


In [ ]:
majory_vote_results = {'Majority_Vote': {}}

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'grade'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'T'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3
key = 'N'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

with open('linear_probing_results_majority_vote_126.json', 'w') as f:


In [ ]:
linear_probing_results = {'Our_CODEX': {}, 'Virtues': {}, 'Our_HE': {}, 'Musk': {}, 'Our_concat': {}}

In [ ]:
linear_probing_results

In [ ]:

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


In [ ]:

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2


key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1)
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1)

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

In [ ]:

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

import numpy as np

# Concatenate he and codex embeddings along last axis
concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))


key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1
#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))

key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_concat'][key] = res_1

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))


key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))

